# 02 — Feature Engineering

Goals:
- Apply and visualise each feature engineering step
- Verify temporal split integrity
- Run the leakage audit
- Save the fitted feature pipeline

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.data.loader import load_raw
from src.data.splitter import temporal_split, get_split_stats
from src.features.cyclic import add_temporal_features
from src.features.geospatial import add_route_distance
from src.features.target_encoding import KFoldTargetEncoder
from src.features.pipeline import FeaturePipeline
from src.utils.leakage_check import run_leakage_audit

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded.')

In [ ]:
# ── Load and split data ────────────────────────────────────────────────────
DATA_PATH = '../data/flights_2023.csv'
ARTIFACTS_DIR = Path('../backend/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

df = load_raw(DATA_PATH)
train_df, val_df, test_df = temporal_split(df)

stats = get_split_stats(train_df, val_df, test_df)
print('Split statistics:')
print(json.dumps(stats, indent=2))

In [ ]:
# ── Visualise cyclic encoding ──────────────────────────────────────────────
sample = train_df.head(500).copy()
sample = add_temporal_features(sample)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(sample['CRS_DEP_TIME_cos'], sample['CRS_DEP_TIME_sin'],
                c=sample['dep_minutes_since_midnight'], cmap='hsv', s=10, alpha=0.6)
axes[0].set_title('Departure Time Cyclic Encoding')
axes[0].set_xlabel('cos'); axes[0].set_ylabel('sin')
axes[0].set_aspect('equal')

axes[1].scatter(sample['DayOfWeek_cos'], sample['DayOfWeek_sin'],
                c=sample['DayOfWeek'], cmap='tab10', s=30, alpha=0.7)
axes[1].set_title('Day of Week Cyclic Encoding')
axes[1].set_xlabel('cos'); axes[1].set_ylabel('sin')
axes[1].set_aspect('equal')

axes[2].scatter(sample['Month_cos'], sample['Month_sin'],
                c=sample['Month'], cmap='plasma', s=30, alpha=0.7)
axes[2].set_title('Month Cyclic Encoding')
axes[2].set_xlabel('cos'); axes[2].set_ylabel('sin')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# ── Route distance distribution ─────────────────────────────────────────────
sample2 = train_df.head(5000).copy()
sample2 = add_route_distance(sample2)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sample2['route_distance_km'].dropna(), bins=60, color='#06b6d4', edgecolor='white')
ax.set_title('Route Distance Distribution (km)')
ax.set_xlabel('Great-circle distance (km)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()
print(f"Missing route_distance_km: {sample2['route_distance_km'].isna().sum()} / {len(sample2)}")

In [ ]:
# ── Full feature pipeline ───────────────────────────────────────────────────
print('Running FeaturePipeline.fit_transform on training data...')
pipeline = FeaturePipeline(smoothing=20.0)
train_transformed = pipeline.fit_transform(train_df, target_col='y')
print(f'Training features shape: {train_transformed.shape}')
print(f'Feature columns: {pipeline.get_feature_names()}')

In [ ]:
# ── Apply to val and test ───────────────────────────────────────────────────
val_transformed  = pipeline.transform(val_df)
test_transformed = pipeline.transform(test_df)
print(f'Val shape: {val_transformed.shape}')
print(f'Test shape: {test_transformed.shape}')

In [ ]:
# ── Feature statistics ──────────────────────────────────────────────────────
feat_cols = pipeline.get_feature_names()
feat_cols_present = [c for c in feat_cols if c in train_transformed.columns]
train_transformed[feat_cols_present].describe().round(3)

In [ ]:
# ── Leakage audit ─────────────────────────────────────────────────────────
audit = run_leakage_audit(train_transformed, target_col='y')
print('Audit result:', audit)

In [ ]:
# ── Save pipeline ────────────────────────────────────────────────────────
pipeline.save(ARTIFACTS_DIR)
print(f'Pipeline saved to {ARTIFACTS_DIR}')

# Save transformed dataframes for notebook 03
train_transformed.to_parquet('../data/train_features.parquet', index=False)
val_transformed.to_parquet('../data/val_features.parquet', index=False)
test_transformed.to_parquet('../data/test_features.parquet', index=False)
print('Feature parquet files saved to data/')